# Granite Speech Demo — full stack in Colab

This notebook spins up the entire [granite-speech-demo](https://github.com/generative-computing/mellea-demos/tree/main/2026-granite-speech) stack inside one Colab runtime — both vLLM model servers (Granite Speech 4.1 STT + Granite Switch 4.1 LLM), the Pipecat backend, and the Next.js frontend — then prints a public URL you open in your browser to start talking.

**Browser mic → WebRTC → Granite Speech STT → Mellea/Granite Switch LLM → Kokoro TTS → browser speaker.**

## Prerequisites

- **GPU runtime: A100 (Colab Pro) recommended.** L4 works. T4 will OOM — both Granite models won't fit.
- **HuggingFace read token.** Free; create one at https://huggingface.co/settings/tokens. Add it as a Colab Secret named `HF_TOKEN` (sidebar → 🔑 → New secret). Used for two things: downloading the Granite model weights, *and* minting per-session WebRTC TURN credentials so audio reaches your browser.
- **Browser:** Chrome, Edge, or Firefox. Safari may behave oddly with WebRTC.

## How long this takes

- **First run on a fresh runtime: ~8–10 min** (model downloads dominate).
- **Subsequent runs with weights cached: ~3 min.**

## What to do

1. Set the `HF_TOKEN` Colab Secret.
2. Switch the runtime to a GPU (Runtime → Change runtime type → A100/L4).
3. **Runtime → Run all.**
4. When the last cell prints a `*.trycloudflare.com` URL, open it, allow mic access, and start talking.

If anything goes wrong, scroll to the bottom — there's a troubleshooting section and a kill-switch cell.

## Cell 2 — Install dependencies (~3 min)

Clones the repo, installs Python deps via `uv`, installs frontend deps via `npm`, and downloads the `cloudflared` binary used for the public tunnel.

In [ ]:
!apt-get -qq install -y nodejs npm
!git clone -q -b notebook https://github.com/psschwei/mellea-demos
%cd mellea-demos/2026-granite-speech
!pip install -q uv
!uv sync --quiet
# vllm is the model server; not a project dep, so install it into the uv env explicitly.
!uv pip install --quiet vllm
!cd frontend && npm install --silent
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Install complete")

## Cell 3 — Configure secrets (instant)

Reads `HF_TOKEN` from Colab Secrets and exports it. Used for both HuggingFace model downloads and per-session TURN credential minting (see [TURN setup](https://turn.fastrtc.org/) — Cloudflare-backed, 10GB/mo free per HF token).

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✅ HF_TOKEN configured — TURN credentials will be minted per-session")

## Cell 4 — Launch vLLM model servers (~5–8 min cold, ~30s cached)

Two vLLM processes:
- **Port 8083:** [`ibm-granite/granite-speech-4.1-2b`](https://huggingface.co/ibm-granite/granite-speech-4.1-2b) — STT.
- **Port 8000:** [`ibm-granite/granite-switch-4.1-3b-preview`](https://huggingface.co/ibm-granite/granite-switch-4.1-3b-preview) — chat LLM with `requirement_check` ALoRA intrinsics.

Both run in the background; logs stream to `logs/vllm-*.log`. The cell blocks until both servers respond on `/v1/models`.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

os.makedirs("logs", exist_ok=True)

speech_log = open("logs/vllm-speech.log", "w")
switch_log = open("logs/vllm-switch.log", "w")

speech_proc = subprocess.Popen(
    [
        "uv", "run", "vllm", "serve", "ibm-granite/granite-speech-4.1-2b",
        "--api-key", "token-abc123",
        "--max-model-len", "2048",
        "--port", "8083",
    ],
    stdout=speech_log, stderr=subprocess.STDOUT,
)
switch_proc = subprocess.Popen(
    [
        "uv", "run", "vllm", "serve", "ibm-granite/granite-switch-4.1-3b-preview",
        "--port", "8000",
    ],
    stdout=switch_log, stderr=subprocess.STDOUT,
)

def wait_for(url: str, name: str, timeout: int = 1200) -> None:
    """Poll until the URL returns 2xx, or raise."""
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if 200 <= r.status < 300:
                    elapsed = int(time.time() - start)
                    print(f"✅ {name} ready ({elapsed}s)")
                    return
        except (urllib.error.URLError, urllib.error.HTTPError, ConnectionError, TimeoutError) as e:
            last_err = e
        time.sleep(5)
    raise TimeoutError(f"{name} did not become ready in {timeout}s. Last error: {last_err}. See logs/vllm-*.log")

print("⏳ Waiting for vLLM servers (downloading weights on first run, ~8 min)...")
wait_for("http://127.0.0.1:8083/v1/models", "Granite Speech (STT)", timeout=1200)
wait_for("http://127.0.0.1:8000/v1/models", "Granite Switch (LLM)", timeout=1200)
print("✅ Both vLLM servers are up")

## Cell 5 — Launch backend + frontend (~30s)

- **Pipecat backend** on port 7860 (FastAPI + SmallWebRTC signaling).
- **Next.js frontend** on port 3000 (proxies WebRTC signaling to the backend in-process).

The backend reads `HF_TOKEN` and uses it to mint a TURN relay credential per session — that's how WebRTC media reaches your browser through the cloudflared tunnel.

In [ ]:
import os
import subprocess

backend_env = {**os.environ}
backend_env.setdefault("HOST", "127.0.0.1")
backend_env.setdefault("PORT", "7860")

backend_log = open("logs/backend.log", "w")
backend_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "granite_speech_demo.server"],
    env=backend_env,
    stdout=backend_log, stderr=subprocess.STDOUT,
)

frontend_env = {**os.environ, "PIPECAT_BACKEND_URL": "http://127.0.0.1:7860"}
frontend_log = open("logs/frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd="frontend",
    env=frontend_env,
    stdout=frontend_log, stderr=subprocess.STDOUT,
)

wait_for("http://127.0.0.1:7860/api/ivr/config", "Pipecat backend", timeout=120)
wait_for("http://127.0.0.1:3000", "Next.js frontend", timeout=120)
print("✅ Backend + frontend are up")

## Cell 6 — Open the public URL and talk

Starts a Cloudflare Quick Tunnel to expose `localhost:3000` on a public `*.trycloudflare.com` URL. The tunnel handles WebRTC *signaling* (HTTP/WebSocket); the *media* path goes through the TURN relay minted by the backend, so audio works even though the Colab runtime has no public IP.

**One tunnel is enough** — the frontend talks to the backend in-process via Next.js API routes.

In [ ]:
import re
import subprocess
import time

tunnel_log_path = "logs/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
public_url = None
deadline = time.time() + 60
while time.time() < deadline and public_url is None:
    time.sleep(2)
    with open(tunnel_log_path) as f:
        m = url_re.search(f.read())
    if m:
        public_url = m.group(0)

if not public_url:
    raise RuntimeError("cloudflared did not print a public URL. See logs/cloudflared.log")

banner = "\n".join([
    "",
    "╔" + "═" * 70 + "╗",
    "║" + "  GRANITE SPEECH DEMO IS LIVE".ljust(70) + "║",
    "╠" + "═" * 70 + "╣",
    "║" + f"  {public_url}".ljust(70) + "║",
    "║" + "".ljust(70) + "║",
    "║" + "  1. Open the URL above in Chrome / Edge / Firefox".ljust(70) + "║",
    "║" + "  2. Allow microphone access when prompted".ljust(70) + "║",
    "║" + "  3. Click the mic button and start talking".ljust(70) + "║",
    "╚" + "═" * 70 + "╝",
    "",
])
print(banner)

## If something goes wrong

Each background process writes to a file in `logs/`:

- `logs/vllm-speech.log` — Granite Speech STT server
- `logs/vllm-switch.log` — Granite Switch LLM server
- `logs/backend.log` — Pipecat backend (look here for TURN minting messages)
- `logs/frontend.log` — Next.js dev server
- `logs/cloudflared.log` — Cloudflare tunnel (the public URL is in here)

View one with `!tail -100 logs/vllm-speech.log` (or open the file from the Colab file browser).

**Common failures:**
- *T4 OOM:* switch the runtime to A100 or L4. Both Granite models won't fit on a T4.
- *`HF_TOKEN` missing:* re-run Cell 3 after adding the secret. Without it, the backend falls back to STUN-only and audio likely won't connect through the cloudflared tunnel.
- *Stuck "waiting for vLLM":* model weights are downloading. The cell waits up to 20 min — let it run.
- *Re-running cells without cleaning up:* old processes still hold the ports. Run the kill-switch cell below, then re-run from the top.

## Caveats

- The `*.trycloudflare.com` URL is public for as long as this notebook runs. Anyone with the link can join the session.
- Colab kernels die after ~24h or when idle. Restart the notebook to get a fresh URL.
- One Colab session serves one user. Each reader runs their own copy of this notebook.

## Kill switch — clean up before re-running

Run this if you need to re-run any of the launch cells. It stops the tunnel, frontend, backend, and both vLLM processes.

In [ ]:
for name, p in [
    ("cloudflared", globals().get("tunnel_proc")),
    ("frontend", globals().get("frontend_proc")),
    ("backend", globals().get("backend_proc")),
    ("vllm-switch", globals().get("switch_proc")),
    ("vllm-speech", globals().get("speech_proc")),
]:
    if p is not None and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=10)
        except Exception:
            p.kill()
        print(f"🛑 stopped {name}")
    else:
        print(f"   {name}: not running")